# Lab02 - Mật mã học hiện đại
**Nhóm 16 - NT101.R11.2**  
**Hà Nguyễn Hiếu An (24520043), Nguyễn Hữu Nghĩa (24521144)**

# Nhiệm vụ 2.1 - Cấu trúc Feistel và sự lan truyền thay đổi
## Mục tiêu 
- Hoàn thiện `feistel_round`, mã hóa `M1 = 0xAB`, `M2 = 0xAC` với `key = 0x12`; in `(L, R)` sau từng vòng và so sánh số bit khác nhau.
- Khối 8 bit, chia hai nửa 4 bit; chạy 4 vòng đúng scaffold.
- $L_i = R_{i-1}$, $R_i = L_{i-1} \oplus F(R_{i-1}, K_i)$; $F(R,K) = (R \oplus K) \mathbin{\&} 0x0F$.
- Khóa con: `[key & 0x0F, (key >> 4) & 0x0F, (key + 1) & 0x0F, (key + 2) & 0x0F]`.
- Ghép bản mã `(L << 4) | R`, **không đổi chỗ thêm ở cuối**, theo hàm mẫu.
- Khoảng cách Hamming: số bit `1` trong `M1 XOR M2`; tỷ lệ thay đổi tính trên 8 bit.

**Đính chính đề:** `0xAB = 10101011`, `0xAC = 10101100`; XOR bằng `00000111`, nên khác **3 bit**, không phải 1 bit. 

In [1]:
def _check_unsigned(value, bits, name):
    if not isinstance(value, int) or isinstance(value, bool) or not 0 <= value < (1 << bits):
        raise ValueError(f'{name} phải là số nguyên không dấu {bits} bit.')


def F(right, subkey):
    _check_unsigned(right, 4, 'right')
    _check_unsigned(subkey, 4, 'subkey')
    return (right ^ subkey) & 0x0F


def feistel_round(L_in, R_in, subkey):
    _check_unsigned(L_in, 4, 'L_in')
    _check_unsigned(R_in, 4, 'R_in')
    return R_in, (L_in ^ F(R_in, subkey)) & 0x0F


def generate_subkeys(key):
    _check_unsigned(key, 8, 'key')
    return [key & 0x0F, (key >> 4) & 0x0F, (key + 1) & 0x0F, (key + 2) & 0x0F]


def feistel_trace(msg, key):
    _check_unsigned(msg, 8, 'msg')
    subkeys = generate_subkeys(key)
    L, R = (msg >> 4) & 0x0F, msg & 0x0F
    states = [(L, R)]
    for subkey in subkeys:
        L, R = feistel_round(L, R, subkey)
        states.append((L, R))
    return states


def feistel_encrypt(msg, key):
    L, R = feistel_trace(msg, key)[-1]
    return (L << 4) | R


def feistel_decrypt(ciphertext, key):
    _check_unsigned(ciphertext, 8, 'ciphertext')
    subkeys = generate_subkeys(key)
    L, R = (ciphertext >> 4) & 0x0F, ciphertext & 0x0F
    for subkey in reversed(subkeys):
        L, R = (R ^ F(L, subkey)) & 0x0F, L
    return (L << 4) | R


def track_avalanche(msg, key):
    states = feistel_trace(msg, key)
    print('Khóa con:', ', '.join(f'{value:04b}' for value in generate_subkeys(key)))
    for index, (L, R) in enumerate(states):
        label = 'Khởi tạo' if index == 0 else f'Vòng {index}'
        print(f'{label}: L={L:04b}, R={R:04b}, ghép=0x{(L << 4) | R:02X}')
    L, R = states[-1]
    return (L << 4) | R


def hamming_distance(first, second):
    _check_unsigned(first, 8, 'first')
    _check_unsigned(second, 8, 'second')
    return (first ^ second).bit_count()


def compare_avalanche(first, second, key):
    first_states, second_states = feistel_trace(first, key), feistel_trace(second, key)
    print(f'XOR đầu vào: {first ^ second:08b}; khác {hamming_distance(first, second)} bit')
    print('Vòng | L1   R1   | L2   R2   | ΔL ΔR | Δ toàn khối | Tỷ lệ')
    distances = []
    for index, ((L1, R1), (L2, R2)) in enumerate(zip(first_states, second_states)):
        left_diff, right_diff = (L1 ^ L2).bit_count(), (R1 ^ R2).bit_count()
        distance = left_diff + right_diff
        distances.append(distance)
        print(f'{index:4} | {L1:04b} {R1:04b} | {L2:04b} {R2:04b} | {left_diff:2} {right_diff:2} | {distance:11} | {distance / 8:6.1%}')
    return distances

## Mã hóa hai bản rõ của đề

In [ ]:
FEISTEL_KEY = 0x12
FEISTEL_M1, FEISTEL_M2 = 0xAB, 0xAC

print('Mã hóa M1 = 0xAB')
feistel_c1 = track_avalanche(FEISTEL_M1, FEISTEL_KEY)
print(f'C1 = 0x{feistel_c1:02X}; giải mã = 0x{feistel_decrypt(feistel_c1, FEISTEL_KEY):02X}')
print('\nMã hóa M2 = 0xAC')
feistel_c2 = track_avalanche(FEISTEL_M2, FEISTEL_KEY)
print(f'C2 = 0x{feistel_c2:02X}; giải mã = 0x{feistel_decrypt(feistel_c2, FEISTEL_KEY):02X}')
print('\nSo sánh theo từng vòng')
feistel_task_distances = compare_avalanche(FEISTEL_M1, FEISTEL_M2, FEISTEL_KEY)

--- Mã hóa M1 = 0xAB ---
Khóa con: 0010, 0001, 0011, 0100
Khởi tạo: L=1010, R=1011, ghép=0xAB
Vòng 1: L=1011, R=0011, ghép=0xB3
Vòng 2: L=0011, R=1001, ghép=0x39
Vòng 3: L=1001, R=1001, ghép=0x99
Vòng 4: L=1001, R=0100, ghép=0x94
C1 = 0x94; giải mã = 0xAB

--- Mã hóa M2 = 0xAC ---
Khóa con: 0010, 0001, 0011, 0100
Khởi tạo: L=1010, R=1100, ghép=0xAC
Vòng 1: L=1100, R=0100, ghép=0xC4
Vòng 2: L=0100, R=1001, ghép=0x49
Vòng 3: L=1001, R=1110, ghép=0x9E
Vòng 4: L=1110, R=0011, ghép=0xE3
C2 = 0xE3; giải mã = 0xAC

--- So sánh theo từng vòng ---
XOR đầu vào: 00000111; khác 3 bit
Vòng | L1   R1   | L2   R2   | ΔL ΔR | Δ toàn khối | Tỷ lệ
   0 | 1010 1011 | 1010 1100 |  0  3 |           3 |  37.5%
   1 | 1011 0011 | 1100 0100 |  3  3 |           6 |  75.0%
   2 | 0011 1001 | 0100 1001 |  3  0 |           3 |  37.5%
   3 | 1001 1001 | 1001 1110 |  0  3 |           3 |  37.5%
   4 | 1001 0100 | 1110 0011 |  3  3 |           6 |  75.0%


## Nhận xét
| Cặp bản rõ | Ban đầu | Vòng 1 | Vòng 2 | Vòng 3 | Vòng 4 |
|---|---:|---:|---:|---:|---:|
| `0xAB`, `0xAC` | 3 | 6 | 3 | 3 | 6 |
| `0xAB`, `0xAA` | 1 | 2 | 1 | 1 | 2 |

- Cặp đề cho bản mã `0x94` và `0xE3`, khác 6/8 bit (75%). Một kết quả cuối khác nhiều bit **không đủ** chứng minh hiệu ứng thác đổ tốt.
- Sự khác biệt lan sang cả hai nửa nhưng **không tăng dần qua mọi vòng**. Cặp khác 1 bit chỉ làm thay đổi 1 hoặc 2 bit đầu ra trong thí nghiệm này.
- Với cùng khóa, phép XOR khóa bị triệt tiêu trong hiệu hai quá trình: $\Delta L_i = \Delta R_{i-1}$, $\Delta R_i = \Delta L_{i-1} \oplus \Delta R_{i-1}$. Cặp sai khác lặp lại sau mỗi 3 vòng; `F` tuyến tính, không trộn các vị trí bit.
- Feistel vẫn giải mã được dù `F` không khả nghịch: $R_{i-1}=L_i$, $L_{i-1}=R_i \oplus F(L_i,K_i)$. Giải mã duyệt khóa con ngược thứ tự.
- Không nên kết luận mô hình đạt tính chất avalanche mạnh. Cần hàm vòng phi tuyến, khuếch tán tốt và nhiều phép thử trên thông điệp/khóa; đây là giới hạn của scaffold, không sửa kết quả để khớp kỳ vọng trong đề.

## Nhập dữ liệu tùy chọn
Đổi `FEISTEL_INTERACTIVE = True` và chạy riêng cell dưới. Nhập bản rõ và khóa dạng Hex trong `00..FF`, có hoặc không có tiền tố `0x`. Mặc định `False` để **Run All** không chờ nhập liệu.

In [4]:
FEISTEL_INTERACTIVE = False

if FEISTEL_INTERACTIVE:
    try:
        feistel_user_msg = int(input('Nhập bản rõ Hex (00..FF): ').strip(), 16)
        feistel_user_key = int(input('Nhập khóa Hex (00..FF): ').strip(), 16)
        feistel_user_ciphertext = track_avalanche(feistel_user_msg, feistel_user_key)
        print(f'Bản mã: 0x{feistel_user_ciphertext:02X}')
        print(f'Giải mã: 0x{feistel_decrypt(feistel_user_ciphertext, feistel_user_key):02X}')
    except ValueError as error:
        print('Lỗi:', error)
else:
    print('Đổi FEISTEL_INTERACTIVE = True để nhập dữ liệu.')

Đổi FEISTEL_INTERACTIVE = True để nhập dữ liệu.


## Kiểm thử
Kiểm tra công thức vòng, khóa con, trạng thái mẫu, sai lệch số bit, chuỗi khoảng cách Hamming, đầu vào không hợp lệ. Kiểm tra khả nghịch từng vòng và mã hóa/giải mã toàn bộ 256 bản rõ với toàn bộ 256 khóa.

In [5]:
import contextlib
import io
import unittest


class TestFeistel(unittest.TestCase):
    def test_round_formula(self):
        self.assertEqual(F(0xB, 0x2), 0x9)
        self.assertEqual(feistel_round(0xA, 0xB, 0x2), (0xB, 0x3))

    def test_subkeys(self):
        self.assertEqual(generate_subkeys(0x12), [2, 1, 3, 4])
        self.assertEqual(generate_subkeys(0xFF), [15, 15, 0, 1])

    def test_known_states(self):
        self.assertEqual(feistel_trace(0xAB, 0x12), [(10, 11), (11, 3), (3, 9), (9, 9), (9, 4)])
        self.assertEqual(feistel_trace(0xAC, 0x12), [(10, 12), (12, 4), (4, 9), (9, 14), (14, 3)])
        self.assertEqual(feistel_encrypt(0xAB, 0x12), 0x94)
        self.assertEqual(feistel_encrypt(0xAC, 0x12), 0xE3)

    def test_input_hamming_distance(self):
        self.assertEqual(hamming_distance(0xAB, 0xAC), 3)
        self.assertEqual(hamming_distance(0xAB, 0xAA), 1)
        self.assertEqual(hamming_distance(0, 255), 8)

    def test_avalanche_sequences(self):
        with contextlib.redirect_stdout(io.StringIO()):
            self.assertEqual(compare_avalanche(0xAB, 0xAC, 0x12), [3, 6, 3, 3, 6])
            self.assertEqual(compare_avalanche(0xAB, 0xAA, 0x12), [1, 2, 1, 1, 2])

    def test_track_output(self):
        captured = io.StringIO()
        with contextlib.redirect_stdout(captured):
            self.assertEqual(track_avalanche(0xAB, 0x12), 0x94)
        self.assertIn('Khởi tạo: L=1010, R=1011', captured.getvalue())
        self.assertIn('Vòng 4: L=1001, R=0100', captured.getvalue())

    def test_every_round_is_reversible(self):
        for subkey in range(16):
            for left in range(16):
                for right in range(16):
                    L, R = feistel_round(left, right, subkey)
                    self.assertEqual(((R ^ F(L, subkey)) & 0x0F, L), (left, right))

    def test_all_messages_and_keys_round_trip(self):
        for key in range(256):
            ciphertexts = set()
            for msg in range(256):
                ciphertext = feistel_encrypt(msg, key)
                ciphertexts.add(ciphertext)
                self.assertEqual(feistel_decrypt(ciphertext, key), msg)
            self.assertEqual(len(ciphertexts), 256)

    def test_invalid_byte_inputs(self):
        for invalid in (-1, 256, True, 1.5, 'AB', None):
            with self.subTest(invalid=invalid):
                for function in (feistel_encrypt, feistel_trace, feistel_decrypt):
                    with self.assertRaises(ValueError):
                        function(invalid, 0x12)
                    with self.assertRaises(ValueError):
                        function(0xAB, invalid)
                with self.assertRaises(ValueError):
                    hamming_distance(invalid, 0)
                with self.assertRaises(ValueError):
                    hamming_distance(0, invalid)

    def test_invalid_nibble_inputs(self):
        for invalid in (-1, 16, True, 1.5, 'B', None):
            with self.subTest(invalid=invalid):
                for args in ((invalid, 0, 0), (0, invalid, 0), (0, 0, invalid)):
                    with self.assertRaises(ValueError):
                        feistel_round(*args)
                with self.assertRaises(ValueError):
                    F(invalid, 0)


feistel_test_result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestFeistel)
)
if not feistel_test_result.wasSuccessful():
    raise RuntimeError('Kiểm thử Feistel thất bại.')

test_all_messages_and_keys_round_trip (__main__.TestFeistel.test_all_messages_and_keys_round_trip) ... 

ok


test_avalanche_sequences (__main__.TestFeistel.test_avalanche_sequences) ... 

ok


test_every_round_is_reversible (__main__.TestFeistel.test_every_round_is_reversible) ... 

...

test_subkeys (__main__.TestFeistel.test_subkeys) ... 

ok


test_track_output (__main__.TestFeistel.test_track_output) ... 

ok


----------------------------------------------------------------------
Ran 10 tests in 3.154s

OK


## Tài liệu tham khảo
- Cấu trúc Feistel: https://en.wikipedia.org/wiki/Feistel_cipher.
- Khoảng cách Hamming: https://en.wikipedia.org/wiki/Hamming_distance.
- `int.bit_count`: https://docs.python.org/3/library/stdtypes.html#int.bit_count.